# Notebook 3 — Avaliação: Métricas, Benchmarking e Exportação ONNX

## Objetivos

1. Avaliar o modelo fine-tunado com métricas de classificação
2. Gerar matriz de confusão e relatório por classe
3. Realizar benchmark de latência (PyTorch vs ONNX Runtime)
4. Exportar modelo para ONNX (Snippet 3 do material)
5. Reproduzir comparação de técnicas PEFT (Tabela 2 do material)

## Conexão Teórica

> *"Após o fine-tuning, exportamos o modelo para formato ONNX, aplicando
> otimizações de grafo computacional como fusão de operadores e constant
> folding."* — Material da Aula 5, Snippet 3

> *"Warm-up strategies pré-executam o modelo com inputs representativos
> antes de aceitar tráfego de produção. [...] Para modelos com sequence
> length variável, recomenda-se warm-up com comprimentos representando
> percentis 50, 90 e 99."*
> — Material da Aula 5, seção 'Estratégias de Warm-up e Caching'

In [ ]:
# Imports
import sys
import os
import json
import warnings

sys.path.insert(0, os.path.abspath(".."))
warnings.filterwarnings("ignore", category=FutureWarning)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch

from src.data_preprocessing import DataPreprocessor
from src.model import PEFTModelWrapper
from src.training import create_datasets
from src.evaluation import (
    calculate_metrics,
    print_classification_report,
    benchmark_latency,
    benchmark_onnx_latency,
    plot_confusion_matrix,
    plot_latency_comparison,
    plot_parameter_comparison,
    create_peft_comparison_table,
)
from src.utils import count_parameters, load_metrics, save_metrics, set_seed, timer

set_seed(42)
sns.set_theme(style="whitegrid")

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE}")

## 1. Carregar Dados e Modelo Treinado

Recarregamos o dataset, o modelo base e os adaptadores LoRA
salvos no Notebook 02.

In [ ]:
# Preparar dados
DATASET_PATH = os.path.join("..", "data", "raw", "dataset.csv")

preprocessor = DataPreprocessor(max_length=128, test_size=0.2, val_size=0.1, random_state=42)
preprocessor.load_data(DATASET_PATH)
preprocessor.clean_data()
preprocessor.prepare_features()
train_df, val_df, test_df = preprocessor.split_data()

# Carregar label map
label_names = sorted(preprocessor.label_map.keys())
print(f"Classes: {label_names}")
print(f"Test set: {len(test_df)} amostras")

In [ ]:
# Carregar modelo com adaptadores LoRA
wrapper = PEFTModelWrapper(
    model_name="distilbert-base-uncased",
    num_labels=preprocessor.num_labels,
    device=DEVICE,
)
base_model = wrapper.load_base_model(quantize_4bit=False)

# Tentar carregar adaptadores LoRA treinados
ADAPTER_PATH = os.path.join("..", "outputs", "models", "lora_adapter_final")
if os.path.exists(ADAPTER_PATH) and os.path.exists(os.path.join(ADAPTER_PATH, "adapter_config.json")):
    from peft import PeftModel
    peft_model = PeftModel.from_pretrained(base_model, ADAPTER_PATH)
    peft_model = peft_model.to(DEVICE)
    print(f"Adaptadores LoRA carregados de {ADAPTER_PATH}")
else:
    # Se adaptadores não existem, aplicar LoRA novo (para demonstração)
    print("Adaptadores não encontrados. Aplicando LoRA novo (sem treinamento).")
    peft_model = wrapper.apply_lora(r=16, lora_alpha=32)

print(f"Parâmetros: {count_parameters(peft_model)}")

## 2. Inferência no Conjunto de Teste

Realizamos predições no conjunto de teste para calcular métricas.

In [ ]:
# Criar dataset de teste
_, _, test_dataset = create_datasets(
    train_df, val_df, test_df,
    tokenizer=wrapper.tokenizer,
    max_length=128,
    text_col="text",
    label_col="label_id",
)

# Inferência
peft_model.eval()
all_preds = []
all_labels = []

with torch.no_grad():
    for i in range(len(test_dataset)):
        sample = test_dataset[i]
        input_ids = sample["input_ids"].unsqueeze(0).to(DEVICE)
        attention_mask = sample["attention_mask"].unsqueeze(0).to(DEVICE)
        label = sample["labels"].item()

        outputs = peft_model(input_ids=input_ids, attention_mask=attention_mask)
        pred = outputs.logits.argmax(dim=-1).item()

        all_preds.append(pred)
        all_labels.append(label)

y_true = np.array(all_labels)
y_pred = np.array(all_preds)
print(f"Inferência concluída: {len(y_true)} amostras")

## 3. Métricas de Classificação

Calculamos accuracy, precision, recall e F1 — métricas padrão
para avaliar modelos fine-tunados com PEFT, conforme benchmarks
GLUE/SuperGLUE citados na Tabela 2 do material (Hu et al., 2022).

In [ ]:
# Calcular métricas completas
metrics = calculate_metrics(y_true, y_pred, label_names=label_names)

print("=" * 50)
print("MÉTRICAS DE CLASSIFICAÇÃO")
print("=" * 50)
print(f"Accuracy:        {metrics['accuracy']:.4f}")
print(f"Precision (macro): {metrics['precision_macro']:.4f}")
print(f"Recall (macro):    {metrics['recall_macro']:.4f}")
print(f"F1 (macro):        {metrics['f1_macro']:.4f}")
print(f"F1 (weighted):     {metrics['f1_weighted']:.4f}")

In [ ]:
# Relatório detalhado por classe
report = print_classification_report(y_true, y_pred, label_names=label_names)
print("\nRelatório de Classificação:")
print(report)

## 4. Matriz de Confusão

In [ ]:
# Matriz de confusão
fig = plot_confusion_matrix(
    y_true, y_pred,
    label_names=label_names,
    title="Matriz de Confusão — DistilBERT + LoRA (r=16)",
    save_path=os.path.join("..", "outputs", "figures", "confusion_matrix.png"),
)
plt.show()

## 5. Benchmark de Latência — PyTorch

Medimos latência de inferência conforme discutido na seção
'Estratégias de Warm-up e Caching de Modelos' do material.

O warm-up elimina overhead de cold start (compilação JIT,
alocação de memória, cache miss).

In [ ]:
# Benchmark de latência PyTorch
sample = test_dataset[0]
input_ids = sample["input_ids"].unsqueeze(0).to(DEVICE)
attention_mask = sample["attention_mask"].unsqueeze(0).to(DEVICE)

pytorch_latency = benchmark_latency(
    peft_model,
    input_ids=input_ids,
    attention_mask=attention_mask,
    num_warmup=5,
    num_runs=50,
)

print("Latência PyTorch (batch=1, seq_len=128):")
print(f"  Média:  {pytorch_latency['mean_ms']:.2f} ms")
print(f"  Std:    {pytorch_latency['std_ms']:.2f} ms")
print(f"  P50:    {pytorch_latency['p50_ms']:.2f} ms")
print(f"  P95:    {pytorch_latency['p95_ms']:.2f} ms")
print(f"  P99:    {pytorch_latency['p99_ms']:.2f} ms")

## 6. Fusão de Pesos e Exportação ONNX

Conforme o **pipeline da Figura 1 do material**:
1. **Merge weights**: W = W₀ + BA (elimina overhead de LoRA)
2. **Export ONNX**: com constant folding e eixos dinâmicos

A exportação ONNX implementa o **Snippet 3 do material**:

```python
# Material da Aula 5, Snippet 3
torch.onnx.export(
    merged_model, dummy_input, "model_optimized.onnx",
    opset_version=17,
    do_constant_folding=True,
    dynamic_axes={'input_ids': {0: 'batch', 1: 'seq'}}
)
```

In [ ]:
# Fundir pesos LoRA (W = W₀ + BA)
# Elimina overhead de adaptadores durante inferência
merged_model = wrapper.merge_lora_weights()
merged_model.eval()
merged_model = merged_model.to(DEVICE)

print(f"Modelo fundido: {count_parameters(merged_model)['total']:,} parâmetros")
print("Adaptadores LoRA mesclados — sem overhead de inferência")

In [ ]:
# Exportar para ONNX (Snippet 3 do material)
ONNX_PATH = os.path.join("..", "outputs", "models", "model_optimized.onnx")

try:
    # Criar um novo wrapper para o modelo fundido e exportar
    os.makedirs(os.path.dirname(ONNX_PATH), exist_ok=True)
    merged_model_cpu = merged_model.to("cpu")
    merged_model_cpu.eval()
    
    dummy_input_ids = torch.randint(0, 1000, (1, 128))
    dummy_attention_mask = torch.ones((1, 128), dtype=torch.long)
    
    torch.onnx.export(
        merged_model_cpu,
        (dummy_input_ids, dummy_attention_mask),
        ONNX_PATH,
        opset_version=17,                   # Opset 17+ recomendado
        do_constant_folding=True,            # Fusão de constantes
        input_names=["input_ids", "attention_mask"],
        output_names=["logits"],
        dynamic_axes={
            "input_ids": {0: "batch", 1: "seq"},
            "attention_mask": {0: "batch", 1: "seq"},
            "logits": {0: "batch"},
        },
    )
    onnx_size_mb = os.path.getsize(ONNX_PATH) / (1024 ** 2)
    print(f"Modelo ONNX exportado: {ONNX_PATH}")
    print(f"Tamanho: {onnx_size_mb:.1f} MB")
    ONNX_EXPORTED = True
except Exception as e:
    print(f"Erro na exportação ONNX: {e}")
    print("Continuando sem benchmark ONNX.")
    ONNX_EXPORTED = False

## 7. Benchmark ONNX Runtime

Comparamos latência PyTorch vs ONNX Runtime, conforme discutido
na **Tabela 1 do material** (Níveis de Otimização ONNX Runtime).

O speedup esperado com otimização de grafo:

$$\text{Speedup} = \frac{T_{eager}}{T_{compiled}} \approx \frac{\sum_{i} (T_{kernel_i} + T_{launch_i} + T_{sync_i})}{\sum_{j} T_{fused\_kernel_j}}$$

In [ ]:
# Benchmark ONNX Runtime
onnx_latency = None

if ONNX_EXPORTED:
    try:
        # Preparar inputs numpy
        np_input_ids = sample["input_ids"].unsqueeze(0).numpy().astype(np.int64)
        np_attention_mask = sample["attention_mask"].unsqueeze(0).numpy().astype(np.int64)

        onnx_latency = benchmark_onnx_latency(
            onnx_path=ONNX_PATH,
            input_ids=np_input_ids,
            attention_mask=np_attention_mask,
            num_warmup=5,
            num_runs=50,
        )

        print("Latência ONNX Runtime (batch=1, seq_len=128):")
        print(f"  Média:  {onnx_latency['mean_ms']:.2f} ms")
        print(f"  Std:    {onnx_latency['std_ms']:.2f} ms")
        print(f"  P50:    {onnx_latency['p50_ms']:.2f} ms")
        print(f"  P95:    {onnx_latency['p95_ms']:.2f} ms")

        speedup = pytorch_latency['mean_ms'] / onnx_latency['mean_ms']
        print(f"\nSpeedup ONNX: {speedup:.2f}x")
    except ImportError:
        print("onnxruntime não instalado. Instale com: pip install onnxruntime")
    except Exception as e:
        print(f"Erro no benchmark ONNX: {e}")
else:
    print("ONNX não exportado. Pulando benchmark.")

In [ ]:
# Visualização de latência
fig = plot_latency_comparison(
    pytorch_latency=pytorch_latency,
    onnx_latency=onnx_latency,
    title="Latência de Inferência — PyTorch vs ONNX Runtime",
    save_path=os.path.join("..", "outputs", "figures", "latency_comparison.png"),
)
plt.show()

## 8. Tabela Comparativa de Técnicas PEFT

Reproduzimos a **Tabela 2 do material** — *Comparação de Técnicas
PEFT para LLaMA-7B* (Hu et al., 2022; Dettmers et al., 2023).

| Técnica | Parâmetros Treináveis | Memória GPU | Latência Adicional | Desempenho* |
|---------|----------------------|-------------|-------------------|-------------|
| Full Fine-Tuning | 7B (100%) | 28GB | 0% | 100% |
| LoRA (r=16) | 4.2M (0.06%) | 8GB | 0% | 97-99% |
| QLoRA (4-bit, r=16) | 4.2M | 6GB | 0% | 96-98% |
| Adapter | 3.5M (0.05%) | 9GB | 5-10% | 95-97% |
| Prefix Tuning | 0.5M (0.007%) | 7GB | 3-5% | 90-95% |
| Prompt Tuning | 0.1M (0.001%) | 7GB | 1-2% | 85-93% |

In [ ]:
# Tabela PEFT comparativa (Tabela 2 do material)
peft_table = create_peft_comparison_table()
print("Tabela 2 — Comparação de Técnicas PEFT para LLaMA-7B")
print("Fonte: Adaptado de Hu et al. (2022) e Dettmers et al. (2023)")
print("=" * 85)
print(peft_table.to_string(index=False))

## 9. Análise de Erros

Identificamos padrões nos erros de classificação para entender
limitações do modelo e guiar melhorias futuras.

In [ ]:
# Análise de erros
test_df_eval = test_df.copy()
test_df_eval["predicted_id"] = y_pred
test_df_eval["correct"] = y_true == y_pred

# Inverter label_map
id_to_label = {v: k for k, v in preprocessor.label_map.items()}
test_df_eval["predicted_label"] = test_df_eval["predicted_id"].map(id_to_label)

errors = test_df_eval[~test_df_eval["correct"]]
print(f"Total de erros: {len(errors)} / {len(test_df_eval)} ({100*len(errors)/len(test_df_eval):.1f}%)")

if len(errors) > 0:
    print(f"\nErros por classe real:")
    print(errors["label"].value_counts())
    print(f"\nExemplos de erros (primeiros 5):")
    for _, row in errors.head(5).iterrows():
        print(f"  Real: {row['label']:12s} → Predito: {row['predicted_label']:12s}")
        print(f"  Texto: {row['text'][:80]}...")
        print()

## 10. Salvar Resultados Finais

In [ ]:
# Consolidar e salvar métricas finais
final_metrics = {
    "classification": metrics,
    "latency_pytorch": pytorch_latency,
    "latency_onnx": onnx_latency,
    "model_info": {
        "base_model": "distilbert-base-uncased",
        "peft_method": "LoRA",
        "lora_r": 16,
        "lora_alpha": 32,
        "target_modules": ["q_lin", "v_lin"],
        "parameters": count_parameters(peft_model),
    },
}

save_metrics(final_metrics, os.path.join("..", "outputs", "logs", "eval_metrics.json"))
print("Métricas finais salvas em outputs/logs/eval_metrics.json")

## Resumo

Neste notebook, avaliamos o modelo fine-tunado e demonstramos:

1. **Métricas**: Accuracy, F1 macro/weighted, precision, recall por classe
2. **Matriz de confusão**: Visualização de erros entre domínios
3. **Fusão de pesos**: W = W₀ + BA elimina overhead de LoRA (Hu et al., 2022)
4. **Exportação ONNX**: Com constant folding e eixos dinâmicos (Snippet 3)
5. **Benchmark de latência**: PyTorch vs ONNX Runtime (Tabela 1 do material)
6. **Comparação PEFT**: Reprodução da Tabela 2 do material

### Referências Principais

- Hu, E.J. et al. (2022). LoRA: Low-Rank Adaptation. ICLR.
- Dettmers, T. et al. (2023). QLoRA. NeurIPS.
- Material da Aula 5, seções 'ONNX' e 'Warm-up e Caching'.

### Vídeos Relacionados

- **Vídeo 3**: Pipeline completo de fine-tuning QLoRA
- **Vídeo 4**: Validação numérica e troubleshooting de exportação ONNX